# SONARA check: EfficientNet-B0 with the same loudness augmentation as B8 (setup C3)

**Why this notebook exists.** The system model B8 (MobileNetV3-Small) was trained with a random loudness change of up to ±12 dB on **every** clip. The runner-up C2 (EfficientNet-B0) was trained with ±6 dB on **20%** of clips. So the final comparison was between two trained setups, not between two architectures under the same training. This notebook trains the missing setup:

| Setup | Model | Loudness augmentation | Everything else |
| :- | :- | :- | :- |
| C2 (already run) | EfficientNet-B0, ImageNet | ±6 dB on 20% of clips | recipe, no normalization |
| **C3 (new)** | EfficientNet-B0, ImageNet | **±12 dB on every clip** | identical to C2 |
| B8 (already run) | MobileNetV3-Small, ImageNet | ±12 dB on every clip | identical to C2 |

C3 vs C2 shows what the augmentation does for EfficientNet-B0. C3 vs B8 is the fair architecture comparison.

## Rules (read before running)

1. **Validation only.** Same locked test set check as the second pass (`bf636617`); no cell trains on, evaluates on, or reads the labels of the locked test set. PVSD is not loaded.
2. **Same splits, seeds and code as the second pass** (seeds 7, 21, 42), so C3 pairs seed by seed with the stored B5, B8 and C2 runs, which are read from `MyDrive/SONARA/run2_second_pass/runs/`.
3. **This is a supplementary check. It does not change the system model.** B8 was selected before the test set was opened; the test set has now been opened, so no model can be re-selected without choosing by test score. Whatever C3 shows is reported in the paper as a check on the selection, and C3 is never evaluated on the test set.
4. **How the result will be read (fixed now, before running).** Compare C3 with B8 on validation ROC-AUC, seed by seed. 0.004 is the run-to-run noise measured in the tuning notebook (identical settings, trials 3 and 23).
   - **C3 clearly higher** (higher on 3 of 3 seeds and mean difference above +0.004): report it openly. B8 stays the system model because it was selected first and is smaller and faster; EfficientNet-B0 with strong loudness augmentation is named as future work.
   - **C3 slightly higher** (higher on 3 of 3 seeds but mean difference within +0.004): report that EfficientNet-B0 is consistently a little higher but within run-to-run noise; the choice of B8 then rests on size and speed.
   - **B8 clearly higher** (higher on 3 of 3 seeds and mean difference above 0.004 in its favour): the selection of B8 is supported on accuracy as well as size and speed.
   - **Otherwise, no clear difference:** under identical training the two architectures perform alike, and the choice of B8 rests on size and speed.
   - The loudness check shows separately whether EfficientNet-B0's weaker robustness in the final evaluation came from its training (C3 as flat as B8) or from the architecture (C3 still not flat).

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Allow Google Drive access. Expect about 1 hour (3 EfficientNet-B0 runs plus loudness checks). Finished runs are skipped on rerun.

## 1. Setup and configuration

In [ ]:
import os, re, copy, glob, hashlib, json, math, random, time
from math import gcd

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, LambdaLR

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__, "| torchvision", torchvision.__version__)

In [ ]:
# ---- Data and outputs ----------------------------------------------------
REPO_URL = "https://github.com/tagalogclark123-cmyk/SONARA-GITHUB.git"
REPO_DIR = "/content/SONARA-GITHUB"
SAVE_TO_DRIVE = True          # keep results in Google Drive so a disconnect loses nothing
OUT_DIR = "/content/drive/MyDrive/SONARA/c3_efficientnet_gain_check" if SAVE_TO_DRIVE else "/content/c3_efficientnet_gain_check"
REFERENCE_RUN_DIR = "/content/drive/MyDrive/SONARA/run2_second_pass/runs"   # stored B5, B8, C2 runs (read only)
SMARTEARS_ROOT = "Smartears"
LABEL_MAP = {"Healthy": "Normal", "None": "Noise", "Sick": "Distress"}

# ---- Protocol --------------------------------------------------------------
LOCK_SEED = 2026              # fixes which 15% of SmartEars is the locked test set
EXPECTED_LOCK_ID = "bf636617" # id of the locked test set made by the first pass
EXPLORE_SEEDS = [7, 21, 42]   # each seed = one train/validation split + one training run per setup
GROUP_BY = "recording"        # clips of one source recording always stay together
TARGET_RECALL = 0.90
LEVEL_OFFSETS_DB = [-20, -10, -6, 0, 6, 10, 20]   # recording-level shifts used in the loudness check

# ---- Input (same as Run 1) -------------------------------------------------
SAMPLE_RATE = 16000
CROP_SEC = 5.0
N_FFT, HOP_LENGTH, N_MELS = 400, 160, 64

# ---- Augmentation (same as Run 1) ------------------------------------------
AUG_SHIFT_PROB, AUG_SHIFT_SEC = 0.5, 0.5
AUG_TIME_MASK_PROB, AUG_FREQ_MASK_PROB = 0.4, 0.4

BATCH_SIZE = 16
LABEL_SMOOTHING = 0.02
DROPOUT = 0.3
USE_PRETRAINED_WEIGHTS = True # False = skip the ImageNet download (the "pretrained" setups then start from random weights)
QUICK_RUN = None              # e.g. 60 = use 60 clips per class, 2 epochs, 2 seeds (code check only)

# ---- The setups to compare -------------------------------------------------
RUN1 = {"norm": "none", "pretrained": False, "optimizer": "adam", "lr": 1e-4, "weight_decay": 1e-4,
        "schedule": "plateau", "max_epochs": 50, "patience": 8, "mixup": 0.0,
        "three_class": False, "gain_prob": 0.2, "gain_db": 6.0}
NORM = {"norm": "per_clip"}
PRETRAINED = {"pretrained": True}
RECIPE = {"optimizer": "adamw", "lr": 3e-4, "weight_decay": 1e-2, "schedule": "cosine",
          "max_epochs": 40, "patience": 10, "mixup": 0.2}
THREE_CLASS = {"three_class": True}
STRONG_GAIN = {"gain_prob": 1.0, "gain_db": 12.0}


def setup(model, parent, change, *overrides):
    cfg = {"model": model, "parent": parent, "change": change, **RUN1}
    for o in overrides:
        cfg.update(o)
    return cfg


EXPERIMENTS = {
    "C3_efficientnet_pre_recipe_gain": setup("efficientnet_b0", "C2_efficientnet_pre_recipe",
                                             "loudness change of ±12 dB on every clip", PRETRAINED, RECIPE, STRONG_GAIN),
}
RERUN_C2 = False              # True = also retrain C2 on the same seeds to measure run-to-run reproducibility (+~45 min)
if RERUN_C2:
    EXPERIMENTS["C2r_efficientnet_pre_recipe_rerun"] = setup("efficientnet_b0", "C2_efficientnet_pre_recipe",
                                                             "rerun of C2 (reproducibility)", PRETRAINED, RECIPE)

# Stored second-pass runs that C3 is compared with (same splits, seeds and code).
REFERENCE_SETUPS = {"B5_mobilenet_pre_recipe": "MobileNetV3-Small, ±6 dB on 20%",
                    "B8_mobilenet_pre_recipe_gain": "MobileNetV3-Small, ±12 dB on every clip (system model)",
                    "C2_efficientnet_pre_recipe": "EfficientNet-B0, ±6 dB on 20% (runner-up)"}
NOISE_AUC = 0.004             # run-to-run noise in validation ROC-AUC, measured in the tuning notebook (trials 3 vs 23)

CLASS_NAMES = ["Non-distress", "Distress"]
SOURCE_INDEX = {"Normal": 0, "Distress": 1, "Noise": 2}   # Distress is index 1 in both the 2- and 3-class models
CROP_SAMPLES = int(round(CROP_SEC * SAMPLE_RATE))
N_FRAMES = CROP_SAMPLES // HOP_LENGTH
NUM_WORKERS = 2 if os.name == "posix" else 0
if QUICK_RUN:
    EXPLORE_SEEDS = EXPLORE_SEEDS[:2]
assert SAVE_TO_DRIVE, "this check needs Drive: the stored B5, B8 and C2 runs are read from it"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
RUN_DIR = os.path.join(OUT_DIR, "runs")
os.makedirs(RUN_DIR, exist_ok=True)
print("Outputs:", OUT_DIR)
print(pd.DataFrame(EXPERIMENTS).T[["model", "parent", "change", "norm", "pretrained", "three_class", "gain_prob", "gain_db"]].to_string())

# Load the stored reference runs and check they were made on the same locked test set and seeds.
REFERENCE = {}
for name in REFERENCE_SETUPS:
    for seed in EXPLORE_SEEDS:
        path = os.path.join(REFERENCE_RUN_DIR, f"{name}_seed{seed}.json")
        assert os.path.exists(path), f"missing reference run {path}: the second-pass outputs must be in Drive"
        r = json.load(open(path))
        assert r["lock_id"] == EXPECTED_LOCK_ID, f"{path} was made on a different locked test set"
        REFERENCE[(name, seed)] = r
c2_cfg = REFERENCE[("C2_efficientnet_pre_recipe", EXPLORE_SEEDS[0])]
for key in ["model", "norm", "pretrained", "optimizer", "lr", "weight_decay", "schedule", "max_epochs", "patience", "mixup"]:
    assert c2_cfg[key] == EXPERIMENTS["C3_efficientnet_pre_recipe_gain"][key], f"C3 differs from C2 in {key}"
print(f"Loaded {len(REFERENCE)} reference runs. C3 differs from C2 only in gain_prob/gain_db: "
      f"C2 {c2_cfg['gain_prob']}/{c2_cfg['gain_db']} -> C3 {STRONG_GAIN['gain_prob']}/{STRONG_GAIN['gain_db']}")

In [ ]:
# Download the repository (code + audio). Takes several minutes the first time.
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
print("Repo folder:", REPO_DIR, "| exists:", os.path.isdir(REPO_DIR))

## 2. Data

SmartEars only, loaded exactly as in Run 1 and the first pass: 16 kHz mono, the whole 5-second clip, binary labels (0 = Non-distress, 1 = Distress). PVSD is not loaded in this notebook.

In [ ]:
SMARTEARS_NAME = re.compile(r"^(rpi\d+)-(\d{8})(\d{6})(?:-(\d+))?$")

rows = []
for folder, source in LABEL_MAP.items():
    for path in sorted(glob.glob(os.path.join(REPO_DIR, SMARTEARS_ROOT, folder, "*.wav"))):
        stem = os.path.splitext(os.path.basename(path))[0]
        m = SMARTEARS_NAME.match(stem)
        recording = f"{m.group(1)}-{m.group(2)}{m.group(3)}" if m else stem
        rows.append({"path": path, "file": f"{folder}/{os.path.basename(path)}", "original_label": folder,
                     "source": source, "label": int(source == "Distress"), "recording": recording})
manifest = pd.DataFrame(rows)
assert len(manifest) > 0, "No audio files found. Check REPO_DIR."
if QUICK_RUN:
    manifest = manifest.groupby("source").head(QUICK_RUN).reset_index(drop=True)
print(manifest.groupby(["original_label", "source", "label"]).size().to_string())


def load_audio(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != SAMPLE_RATE:
        g = gcd(int(sr), SAMPLE_RATE)
        wav = resample_poly(wav, SAMPLE_RATE // g, sr // g).astype(np.float32)
    return wav


t0 = time.time()
clips = []
for i, path in enumerate(manifest["path"]):
    clips.append(np.clip(load_audio(path) * 32767.0, -32768, 32767).astype(np.int16))
    if (i + 1) % 1000 == 0:
        print(f"  loaded {i + 1}/{len(manifest)} clips ({time.time() - t0:.0f}s)")
print(f"Loaded {len(clips)} clips in {time.time() - t0:.0f}s")

## 3. Recompute the locked test set, then split what is left

The locked test set is recomputed from `LOCK_SEED` and must have the same id as in the first pass; the cell stops otherwise. From this cell on, the code only ever indexes `dev_idx`, the other 85%.

For each exploration seed, the development data is split again into training (70% of all clips) and validation (15% of all clips). Both splits are stratified by source category and grouped by source recording.

In [ ]:
def grouped_folds(indices, n_splits, seed):
    df = manifest.loc[indices]
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_of = np.zeros(len(df), dtype=int)
    for fold, (_, idx) in enumerate(sgkf.split(df, df["source"], df[GROUP_BY])):
        fold_of[idx] = fold
    return fold_of


all_idx = manifest.index.to_numpy()
fold_of = grouped_folds(all_idx, 20, LOCK_SEED)
locked_test_idx = all_idx[fold_of >= 17]          # 3 of 20 folds = 15%
dev_idx = all_idx[fold_of < 17]                   # the only clips this notebook uses
assert not set(manifest.loc[locked_test_idx, GROUP_BY]) & set(manifest.loc[dev_idx, GROUP_BY]), "group leakage"

lock_path = os.path.join(OUT_DIR, "locked_test_set.csv")
manifest.loc[locked_test_idx, ["file", "recording"]].to_csv(lock_path, index=False)
lock_id = hashlib.md5("\n".join(sorted(manifest.loc[locked_test_idx, "file"])).encode()).hexdigest()[:8]
print(f"Locked test set: {len(locked_test_idx)} clips, id {lock_id}, saved to {lock_path}")
print(f"Development set: {len(dev_idx)} clips")
if not QUICK_RUN:
    assert lock_id == EXPECTED_LOCK_ID, (f"Locked test set id {lock_id} differs from the first pass ({EXPECTED_LOCK_ID}). "
                                         "Stop: results would not be comparable, and first-pass test clips could leak in.")
DEV = set(dev_idx.tolist())
del locked_test_idx                                # nothing below can reach the locked clips by accident


def make_split(seed):
    fold_of = grouped_folds(dev_idx, 17, seed)
    split = {"train": dev_idx[fold_of < 14], "val": dev_idx[fold_of >= 14]}
    assert not set(manifest.loc[split["train"], GROUP_BY]) & set(manifest.loc[split["val"], GROUP_BY]), "group leakage"
    assert set(split["train"].tolist()) | set(split["val"].tolist()) <= DEV
    return split


for s in EXPLORE_SEEDS:
    sp = make_split(s)
    print(f"seed {s}: train {len(sp['train'])} {np.bincount(manifest.loc[sp['train'], 'label']).tolist()} | "
          f"val {len(sp['val'])} {np.bincount(manifest.loc[sp['val'], 'label']).tolist()}")

## 4. Features, normalization and augmentation

The features are the same as in Run 1: mel power in dB plus delta and delta-delta, 3 × 64 × 500.

**Normalization (new).** With `norm = "per_clip"`, each channel of each clip has its own mean subtracted and is divided by its own standard deviation. A clip recorded louder or quieter then looks the same to the model. It uses only the clip itself, so no statistics are carried from one split to another.

Augmentation is applied before normalization. It is unchanged from the first pass except in setup B8, where every training clip gets a random loudness change of up to ±12 dB. Mixup, part of the recipe, blends each training clip with another one from the same batch and blends their labels in the same proportion.

In [ ]:
class LogMelDeltas(nn.Module):
    """Waveform (batch, samples) -> raw dB features (batch, 3, n_mels, frames)."""

    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                                        n_mels=N_MELS, center=True, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.deltas = torchaudio.transforms.ComputeDeltas()

    def features(self, wav):
        db = self.to_db(self.mel(wav))
        d1 = self.deltas(db)
        return torch.stack([db, d1, self.deltas(d1)], dim=1)

    def forward(self, wav, lengths):
        x = self.features(wav)[..., :N_FRAMES]
        for i in torch.nonzero(lengths < CROP_SAMPLES).flatten().tolist():   # clips a few ms short of 5 s
            f = self.features(wav[i:i + 1, :max(int(lengths[i]), N_FFT)])[0]
            t = min(f.shape[-1], N_FRAMES)
            x[i, 0] = f[0].min()
            x[i, 1:] = 0.0
            x[i, :, :, :t] = f[:, :, :t]
        return x


featurizer = LogMelDeltas().to(device).eval()


def normalize(x, mode):
    if mode == "per_clip":
        mean = x.mean(dim=(2, 3), keepdim=True)
        std = x.std(dim=(2, 3), keepdim=True)
        return (x - mean) / (std + 1e-5)
    return x


def augment_features(x, cfg):
    _, _, n_freq, n_time = x.shape
    max_shift = int(AUG_SHIFT_SEC * SAMPLE_RATE / HOP_LENGTH)
    for i in range(x.size(0)):
        if max_shift and random.random() < AUG_SHIFT_PROB:
            x[i] = torch.roll(x[i], random.randint(-max_shift, max_shift), dims=2)
        ch_mean = x[i].mean(dim=(1, 2), keepdim=True)
        if random.random() < AUG_TIME_MASK_PROB:
            width = random.randint(5, n_time // 8)
            t0 = random.randint(0, n_time - width)
            x[i, :, :, t0:t0 + width] = ch_mean
        if random.random() < AUG_FREQ_MASK_PROB:
            height = random.randint(3, n_freq // 8)
            f0 = random.randint(0, n_freq - height)
            x[i, :, f0:f0 + height, :] = ch_mean
        if random.random() < cfg["gain_prob"]:
            x[i, 0] += random.uniform(-cfg["gain_db"], cfg["gain_db"])
    return x


class ClipDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)
        assert set(self.indices.tolist()) <= DEV, "a locked test clip was requested"
        self.labels = manifest["label"].to_numpy()
        self.sources = manifest["source"].map(SOURCE_INDEX).to_numpy()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        seg = clips[idx][:CROP_SAMPLES].astype(np.float32) / 32768.0
        n = len(seg)
        if n < CROP_SAMPLES:
            seg = np.pad(seg, (0, CROP_SAMPLES - n))
        return torch.from_numpy(seg), int(self.labels[idx]), int(self.sources[idx]), n


def make_loader(indices, train, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ClipDataset(indices), batch_size=BATCH_SIZE, shuffle=train, num_workers=NUM_WORKERS,
                      generator=g, pin_memory=(device.type == "cuda"))

## 5. Models

- **CNN–BiLSTM**: the Run 1 model, unchanged (1,221,186 parameters).
- **MobileNetV3-Small**: about 1.5 million parameters. With `pretrained = True` it starts from ImageNet weights and only its last layer is new.
- **EfficientNet-B0**: about 4 million parameters, ImageNet weights. Larger, but still a size that runs on a Raspberry Pi 4.

ImageNet is a photo dataset. Its weights are used only as a starting point; every layer is then trained on SmartEars. The three feature channels (dB, delta, delta-delta) take the place of the three colour channels.

In the three-class setups the last layer has three outputs (Healthy, Sick, None) and the Sick probability is the distress score.

In [ ]:
class CNN_BiLSTM(nn.Module):
    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, lstm_hidden=128, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.lstm = nn.LSTM(input_size=cnn_channels * (n_mels // 4), hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        b, c, fq, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * fq)
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(torch.cat([h_n[-2], h_n[-1]], dim=1)))


def build_model(name, pretrained, num_classes=2):
    use_weights = "IMAGENET1K_V1" if (pretrained and USE_PRETRAINED_WEIGHTS) else None
    if name == "cnn_bilstm":
        return CNN_BiLSTM(num_classes=num_classes)
    if name == "mobilenetv3_small":
        m = torchvision.models.mobilenet_v3_small(weights=use_weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    if name == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights=use_weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    raise ValueError(name)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def time_ms(fn, runs=20):
    for _ in range(3):
        fn()
    ts = []
    for _ in range(runs):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    return round(float(np.median(ts)), 1)


# Size and a CPU timing proxy (this Colab CPU, NOT the Raspberry Pi 4) for the deployability check.
x_cpu = torch.randn(1, 3, N_MELS, N_FRAMES)
deploy = []
with torch.no_grad():
    for name in ["mobilenetv3_small", "efficientnet_b0"]:
        mdl = build_model(name, pretrained=False).eval()
        deploy.append({"model": name, "parameters": count_params(mdl), "size_MB": round(count_params(mdl) * 4 / 1e6, 2),
                       "cpu_ms_per_clip": time_ms(lambda: mdl(x_cpu))})
deploy = pd.DataFrame(deploy)
deploy.to_csv(os.path.join(OUT_DIR, "model_size_and_cpu_time.csv"), index=False)
print(f"CPU threads: {torch.get_num_threads()} (Colab proxy, not the Pi)")
print(deploy.to_string(index=False))

## 6. Training and the loudness check

Training is the same as in the first pass: weighted cross-entropy with label smoothing of 0.02, and the best checkpoint is the epoch with the lowest validation loss. Every setup in this pass uses the recipe (AdamW, learning rate 3e-4 with warm-up and cosine decay, mixup, at most 40 epochs).

**Loudness check.** After training, the best checkpoint scores its validation clips again with the level shifted by each value in `LEVEL_OFFSETS_DB`. A shift of +10 dB is what the same sound would look like through a microphone or gain setting about three times more sensitive. The threshold is the one chosen at the original level and is **not** re-tuned, because a deployed gateway cannot re-tune it. A robust model keeps roughly the same recall and false-alarm rate at every shift.

In [ ]:
def forward_features(wav, n, cfg, training=False, level_offset_db=0.0):
    with torch.no_grad():
        x = featurizer(wav, n)
        if level_offset_db:
            x[:, 0] += level_offset_db                # a level change is a constant offset on the dB channel
        if training:
            x = augment_features(x, cfg)
        return normalize(x, cfg["norm"])


def run_epoch(model, loader, cfg, criterion, optimizer=None, level_offset_db=0.0):
    training = optimizer is not None
    model.train(training)
    total_loss, total, probs, labels = 0.0, 0, [], []
    with torch.set_grad_enabled(training):
        for wav, y, src, n in loader:
            wav, y, src = wav.to(device), y.to(device), src.to(device)
            target = src if cfg["three_class"] else y
            x = forward_features(wav, n, cfg, training, level_offset_db)
            if training and cfg["mixup"] > 0:
                lam = float(np.random.beta(cfg["mixup"], cfg["mixup"]))
                perm = torch.randperm(x.size(0), device=device)
                logits = model(lam * x + (1 - lam) * x[perm])
                loss = lam * criterion(logits, target) + (1 - lam) * criterion(logits, target[perm])
            else:
                logits = model(x)
                loss = criterion(logits, target)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            else:
                probs.append(F.softmax(logits, dim=1)[:, 1].cpu().numpy())   # index 1 = Distress
                labels.append(y.cpu().numpy())
            total_loss += loss.item() * y.size(0)
            total += y.size(0)
    if training:
        return total_loss / total, None, None
    return total_loss / total, np.concatenate(probs), np.concatenate(labels)


def train_setup(name, seed, split):
    cfg = EXPERIMENTS[name]
    max_epochs = 2 if QUICK_RUN else cfg["max_epochs"]
    n_classes = 3 if cfg["three_class"] else 2
    set_seed(seed)
    model = build_model(cfg["model"], cfg["pretrained"], n_classes).to(device)
    targets = (manifest.loc[split["train"], "source"].map(SOURCE_INDEX) if cfg["three_class"]
               else manifest.loc[split["train"], "label"])
    counts = np.bincount(targets, minlength=n_classes)
    weights = torch.tensor(counts.mean() / counts, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
    if cfg["optimizer"] == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if cfg["schedule"] == "cosine":
        warm = 2
        scheduler = LambdaLR(optimizer, lambda e: (e + 1) / warm if e < warm else
                             max(0.02, 0.5 * (1 + math.cos(math.pi * (e - warm) / max(1, max_epochs - warm)))))
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)
    train_loader = make_loader(split["train"], True, seed)
    val_loader = make_loader(split["val"], False, seed)

    history, best = [], {"loss": float("inf")}
    stale = 0
    for epoch in range(1, max_epochs + 1):
        train_loss, _, _ = run_epoch(model, train_loader, cfg, criterion, optimizer)
        val_loss, p_val, y_val = run_epoch(model, val_loader, cfg, criterion)
        scheduler.step(val_loss) if cfg["schedule"] == "plateau" else scheduler.step()
        improved = val_loss < best["loss"] - 1e-4
        if improved:
            best, stale = {"loss": val_loss, "epoch": epoch, "p_val": p_val, "y_val": y_val,
                           "state": copy.deepcopy(model.state_dict())}, 0
        else:
            stale += 1
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_auc": roc_auc_score(y_val, p_val), "best": improved})
        print(f"  [{name} seed {seed}] epoch {epoch:02d} train {train_loss:.4f} val {val_loss:.4f} "
              f"auc {history[-1]['val_auc']:.4f}{' *' if improved else ''}")
        if stale >= cfg["patience"]:
            break
    model.load_state_dict(best.pop("state"))
    return model.eval(), best, pd.DataFrame(history), val_loader, criterion


def level_check(model, cfg, val_loader, criterion, threshold):
    """Validation ROC-AUC, recall and false-alarm rate at each level shift, threshold held fixed."""
    out = {}
    for off in LEVEL_OFFSETS_DB:
        _, p, y = run_epoch(model, val_loader, cfg, criterion, level_offset_db=float(off))
        flagged = (p >= threshold).astype(int)
        out[str(off)] = {"auc": roc_auc_score(y, p), "recall": float(flagged[y == 1].mean()),
                         "false_alarms": float(flagged[y == 0].mean())}
    return out


def choose_threshold(y, p, target=TARGET_RECALL):
    ok = [t for t in np.round(np.arange(0.01, 0.995, 0.005), 3)
          if recall_score(y, (p >= t).astype(int), zero_division=0) >= target]
    return float(max(ok)) if ok else 0.5


def validation_metrics(y, p, sources):
    pred = (p >= 0.5).astype(int)
    thr = choose_threshold(y, p)
    flagged = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, flagged, labels=[0, 1]).ravel()
    return {"val_roc_auc": roc_auc_score(y, p), "val_pr_auc": average_precision_score(y, p),
            "val_accuracy_at_0.5": accuracy_score(y, pred),
            "val_distress_f1_at_0.5": f1_score(y, pred, zero_division=0),
            "val_distress_recall_at_0.5": recall_score(y, pred, zero_division=0),
            "threshold_for_recall_0.90": thr,
            "val_false_alarm_rate_at_recall_0.90": fp / max(fp + tn, 1),
            "val_false_alarms_normal": float(flagged[sources == "Normal"].mean()),
            "val_false_alarms_noise": float(flagged[sources == "Noise"].mean())}

## 7. Run every setup on every seed

Each finished run writes a small JSON file, its validation predictions and its training history to `OUT_DIR/runs`. Finished runs are skipped when this cell is run again, so after a disconnect just run the notebook again.

In [ ]:
for seed in EXPLORE_SEEDS:
    split = make_split(seed)
    for name, cfg in EXPERIMENTS.items():
        base = os.path.join(RUN_DIR, f"{name}_seed{seed}")
        if os.path.exists(base + ".json"):
            print(f"skip (done): {name}_seed{seed}")
            continue
        t0 = time.time()
        model, best, history, val_loader, criterion = train_setup(name, seed, split)
        sources = manifest.loc[split["val"], "source"].to_numpy()
        metrics = validation_metrics(best["y_val"], best["p_val"], sources)
        levels = level_check(model, cfg, val_loader, criterion, metrics["threshold_for_recall_0.90"])
        result = {"setup": name, "seed": seed, "lock_id": lock_id,
                  **{k: v for k, v in cfg.items()}, "parameters": count_params(model),
                  "epochs_run": len(history), "best_epoch": best["epoch"], "best_val_loss": best["loss"],
                  "train_minutes": (time.time() - t0) / 60, **metrics, "level_check": levels}
        np.savez(base + "_val_predictions.npz", p_val=best["p_val"], y_val=best["y_val"], rows_val=split["val"])
        history.to_csv(base + "_history.csv", index=False)
        with open(base + ".json", "w") as f:        # written last: marks the run as complete
            json.dump(result, f, indent=1, default=float)
        print(f"DONE {name}_seed{seed}: val AUC {result['val_roc_auc']:.4f} | acc@0.5 {result['val_accuracy_at_0.5']:.4f} | "
              f"false alarms at 0.90 recall {result['val_false_alarm_rate_at_recall_0.90']:.4f} | "
              f"recall at -10/0/+10 dB {levels['-10']['recall']:.2f}/{levels['0']['recall']:.2f}/{levels['10']['recall']:.2f} | "
              f"{result['train_minutes']:.1f} min\n")
        del model

## 8. Validation results: C3 next to the stored B5, B8 and C2 runs

All numbers are **validation** results on the same three splits (mean ± SD over seeds 7, 21, 42). Nothing here touches the locked test set.

In [ ]:
def pm(s, digits=4):
    s = pd.Series(s, dtype=float)
    return f"{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}" if len(s) > 1 else f"{s.mean():.{digits}f}"


new_runs = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(os.path.join(RUN_DIR, "*.json")))])
new_runs = new_runs[new_runs["setup"].isin(EXPERIMENTS) & new_runs["seed"].isin(EXPLORE_SEEDS) & (new_runs["lock_id"] == lock_id)]
ref_runs = pd.DataFrame([dict(r, setup=n) for (n, s), r in REFERENCE.items()])
runs = pd.concat([ref_runs, new_runs], ignore_index=True)
runs.drop(columns="level_check").to_csv(os.path.join(OUT_DIR, "c3_all_runs.csv"), index=False)

order = list(REFERENCE_SETUPS) + list(EXPERIMENTS)
summary = []
for name in order:
    g = runs[runs["setup"] == name].sort_values("seed")
    if len(g) == 0:
        continue
    summary.append({"Setup": name, "Seeds": len(g), "gain": f"±{g['gain_db'].iloc[0]:.0f} dB on {g['gain_prob'].iloc[0]:.0%}",
                    "ROC-AUC": pm(g["val_roc_auc"]), "ROC-AUC per seed": " / ".join(f"{v:.4f}" for v in g["val_roc_auc"]),
                    "Accuracy @0.5": pm(g["val_accuracy_at_0.5"], 3),
                    "Distress F1 @0.5": pm(g["val_distress_f1_at_0.5"], 3),
                    "False alarms @0.90 recall": pm(g["val_false_alarm_rate_at_recall_0.90"], 3),
                    "…on Normal": pm(g["val_false_alarms_normal"], 3), "…on Noise": pm(g["val_false_alarms_noise"], 3),
                    "Best epoch": pm(g["best_epoch"], 1)})
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_DIR, "c3_validation_summary.csv"), index=False)
pd.set_option("display.width", 250)
print("VALIDATION results (not test results)")
print(summary.to_string(index=False))

## 9. Seed-by-seed comparisons

Same seed = same train/validation split, so differences are paired. A change counts only if it is better on most seeds, and a ROC-AUC difference within ±0.004 (run-to-run noise) is reported as no clear difference.

- **C3 vs C2:** what the strong loudness augmentation does for EfficientNet-B0.
- **B8 vs B5:** what the same augmentation did for MobileNetV3-Small (for comparison).
- **C3 vs B8:** the fair architecture comparison under identical training.

In [ ]:
def per_seed(name):
    g = runs[runs["setup"] == name]
    return {int(r["seed"]): r for _, r in g.iterrows()}


PAIRS = [("C3_efficientnet_pre_recipe_gain", "C2_efficientnet_pre_recipe", "gain effect on EfficientNet-B0"),
         ("B8_mobilenet_pre_recipe_gain", "B5_mobilenet_pre_recipe", "gain effect on MobileNetV3-Small"),
         ("C3_efficientnet_pre_recipe_gain", "B8_mobilenet_pre_recipe_gain", "EfficientNet-B0 vs MobileNetV3-Small, same training")]
if RERUN_C2:
    PAIRS.append(("C2r_efficientnet_pre_recipe_rerun", "C2_efficientnet_pre_recipe", "reproducibility of C2"))

comparisons = []
for a_name, b_name, label in PAIRS:
    a, b = per_seed(a_name), per_seed(b_name)
    common = sorted(set(a) & set(b))
    if not common:
        continue
    row = {"Comparison": label, "A": a_name, "B": b_name, "Seeds": len(common)}
    for key, short, lower_better in [("val_roc_auc", "ROC-AUC", False), ("val_accuracy_at_0.5", "Accuracy @0.5", False),
                                     ("val_false_alarm_rate_at_recall_0.90", "False alarms @0.90", True)]:
        d = np.array([a[s][key] - b[s][key] for s in common])
        better = int((d < 0).sum()) if lower_better else int((d > 0).sum())
        row[f"{short}: A − B"] = round(float(d.mean()), 4)
        row[f"{short}: seeds A better"] = better
    comparisons.append(row)
comparisons = pd.DataFrame(comparisons)
comparisons.to_csv(os.path.join(OUT_DIR, "c3_comparisons.csv"), index=False)
print(comparisons.to_string(index=False))

## 10. Loudness check

Threshold set at 0 dB to catch 90% of Distress clips, then held fixed while the level is shifted. If C3 is about as flat as B8, EfficientNet-B0's weaker robustness in the final evaluation came from its training, not its architecture.

In [ ]:
level_rows = []
for name in order:
    g = runs[runs["setup"] == name]
    if len(g) == 0:
        continue
    row = {"Setup": name}
    for metric, label in [("recall", "Recall"), ("false_alarms", "False alarms"), ("auc", "ROC-AUC")]:
        for off in LEVEL_OFFSETS_DB:
            row[f"{label} {off:+d} dB"] = round(float(np.mean([lc[str(off)][metric] for lc in g["level_check"]])), 3)
    for span in (10, 20):
        row[f"Largest recall change at ±{span} dB"] = round(max(abs(row[f"Recall {o:+d} dB"] - row["Recall +0 dB"]) for o in (-span, span)), 3)
        row[f"Largest false-alarm change at ±{span} dB"] = round(max(abs(row[f"False alarms {o:+d} dB"] - row["False alarms +0 dB"]) for o in (-span, span)), 3)
    level_rows.append(row)
level_table = pd.DataFrame(level_rows)
level_table.to_csv(os.path.join(OUT_DIR, "c3_level_check.csv"), index=False)
for label in ["Recall", "False alarms", "ROC-AUC"]:
    print(f"{label} at each level shift (threshold fixed at its 0 dB value):")
    print(level_table[["Setup"] + [f"{label} {off:+d} dB" for off in LEVEL_OFFSETS_DB]].to_string(index=False), "\n")
print(level_table[["Setup"] + [c for c in level_table.columns if c.startswith("Largest")]].to_string(index=False))

## 11. Reading the result (rule fixed in Part 0)

In [ ]:
a, b = per_seed("C3_efficientnet_pre_recipe_gain"), per_seed("B8_mobilenet_pre_recipe_gain")
common = sorted(set(a) & set(b))
if len(common) < len(EXPLORE_SEEDS):
    print("Not all seeds finished: rerun the notebook before reading the result.")
else:
    d = np.array([a[s]["val_roc_auc"] - b[s]["val_roc_auc"] for s in common])
    n = len(common)
    print(f"C3 − B8 validation ROC-AUC per seed: {np.round(d, 4).tolist()} (mean {d.mean():+.4f}; noise level {NOISE_AUC})")
    if (d > 0).sum() == n and d.mean() > NOISE_AUC:
        verdict = ("C3 CLEARLY HIGHER than B8. Report this openly. B8 stays the system model (selected before the test set "
                   "was opened; smaller and faster). Name EfficientNet-B0 with strong loudness augmentation as future work.")
    elif (d > 0).sum() == n:
        verdict = ("C3 SLIGHTLY HIGHER than B8 on every seed, but within run-to-run noise. Report the small, consistent "
                   "difference; the choice of B8 rests on size and speed.")
    elif (d < 0).sum() == n and -d.mean() > NOISE_AUC:
        verdict = "B8 CLEARLY HIGHER than C3. The selection of B8 is supported on accuracy as well as size and speed."
    else:
        verdict = ("NO CLEAR DIFFERENCE between C3 and B8. Under identical training the two architectures perform alike; "
                   "the choice of B8 rests on size and speed.")
    print(verdict)
    print("Do not evaluate C3 on the test set and do not change the system model.")
lt = level_table.set_index("Setup")
for name in ["C2_efficientnet_pre_recipe", "C3_efficientnet_pre_recipe_gain", "B8_mobilenet_pre_recipe_gain"]:
    if name in lt.index:
        print(f"{name}: recall at -20 / 0 / +20 dB = {lt.loc[name, 'Recall -20 dB']:.3f} / "
              f"{lt.loc[name, 'Recall +0 dB']:.3f} / {lt.loc[name, 'Recall +20 dB']:.3f}")
print("\nThese are validation results. The locked test set (id " + lock_id + ") and PVSD were not used.")

## 12. What to do with these results

1. Copy the files in `MyDrive/SONARA/c3_efficientnet_gain_check/` (the four `c3_*.csv` files and `model_size_and_cpu_time.csv`) and this executed notebook into the repo under `results/c3_efficientnet_gain_check/`.
2. Report C3 in the paper as a supplementary validation check on the model selection, with the comparison and loudness tables. Do not report it as a test result.
3. Do not evaluate C3 on the locked test set and do not change the system model.